In [18]:
import numpy as np
import pandas as pd

In [19]:
X_train = pd.read_csv('../data/processed/X_train.csv',index_col=0)
X_test = pd.read_csv('../data/processed/X_test.csv',index_col=0)
y_train = pd.read_csv('../data/processed/y_train.csv',index_col=0)
y_test = pd.read_csv('../data/processed/y_test.csv',index_col=0)

In [26]:
test = pd.read_csv('../data/raw/test.csv')

In [22]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer  # Required for IterativeImputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.preprocessing import FunctionTransformer, RobustScaler, OneHotEncoder, OrdinalEncoder
import sklearn

# Force scikit-learn to output Pandas DataFrames at every step
sklearn.set_config(transform_output="pandas")

class StringExtractionTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = np.array(X.columns, dtype=object)
        return self
    
    def transform(self, X):
        X = X.copy()
        
        # 1. Clean ALL object/string columns so missing values are unified to true np.nan
        for col in X.select_dtypes(include=['object', 'string']).columns:
            X[col] = X[col].replace({'nan': np.nan, 'None': np.nan, '': np.nan})
            X[col] = X[col].mask(X[col].isna(), np.nan)

        # 2. String extractions
        if 'PassengerId' in X.columns:
            X['Passengerno'] = X['PassengerId'].astype(str).str.split('_').str.get(1)
            X['Passengerno'] = X['Passengerno'].replace({'nan': np.nan, 'None': np.nan})
            
        if 'Cabin' in X.columns:
            X[['Deck', 'Num', 'Side']] = X['Cabin'].astype(str).str.split('/', expand=True)
            X['Num'] = pd.to_numeric(X['Num'], errors='coerce')
            X[['Deck', 'Side']] = X[['Deck', 'Side']].replace({'nan': np.nan, 'None': np.nan})
            
        return X
        

    def get_feature_names_out(self, input_features=None):
        # Fallback to saved feature names if Scikit-learn doesn't pass them
        if input_features is None:
            input_features = getattr(self, "feature_names_in_", [])
        
        cols = list(input_features)
        
        # Append the new columns we created in transform()
        if 'PassengerId' in cols and 'Passengerno' not in cols:
            cols.append('Passengerno')
            
        if 'Cabin' in cols:
            for new_col in ['Deck', 'Num', 'Side']:
                if new_col not in cols:
                    cols.append(new_col)
                    
        return np.array(cols, dtype=object)


# ---------------------------------------------------------
# 2. Custom Transformer for Feature Engineering (TotalBill & AgeGroup)
# ---------------------------------------------------------
class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = np.array(X.columns, dtype=object)
        return self
    
    def transform(self, X):
        X = X.copy()
        
        # TotalBill
        bill_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
        existing_bill_cols = [c for c in bill_cols if c in X.columns]
        
        if existing_bill_cols:
            # Ensure they are numeric after imputation
            X[existing_bill_cols] = X[existing_bill_cols].apply(pd.to_numeric, errors='coerce')
            X['TotalBill'] = X[existing_bill_cols].sum(axis=1)
        
        # AgeGroup
        bins = [0, 12, 19, 35, 59, 150]
        labels = ['Child', 'Teenager', 'Young Adult', 'Adult', 'Senior']
        
        if 'Age' in X.columns:
            # Impute or fill empty bins BEFORE converting to string
            age_series = pd.to_numeric(X['Age'], errors='coerce')
            
            X['AgeGroup'] = pd.cut(
                age_series, 
                bins=bins, 
                labels=labels, 
                right=True, 
                include_lowest=True
            )
            # Use a default category like most frequent ("Adult") or fill properly instead of creating a "None" string
            X['AgeGroup'] = X['AgeGroup'].astype(object).fillna("Adult").astype(str)
            
        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = getattr(self, "feature_names_in_", [])
        
        cols = list(input_features)
        bill_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
        
        # Append TotalBill if the billing columns existed in the input
        if any(c in cols for c in bill_cols) and 'TotalBill' not in cols:
            cols.append('TotalBill')
            
        # Append AgeGroup if Age existed in the input
        if 'Age' in cols and 'AgeGroup' not in cols:
            cols.append('AgeGroup')
            
        return np.array(cols, dtype=object)

def safe_log1p(X):
    X = np.asarray(X, dtype=np.float64)
    return np.log1p(np.nan_to_num(X))

    
# (Make sure you define these lists based on your raw dataset columns)
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side', 'Passengerno'] 
# Add 'Num' back here
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Num'] 

# Add 'CryoSleep' back here
ordinal_cols = ['VIP', 'Side', 'CryoSleep']
ohe_cols = ['HomePlanet', 'Destination', 'Deck', 'AgeGroup', 'Passengerno']

# ---------------------------------------------------------
# 4. Building the Individual Pipeline Steps
# ---------------------------------------------------------
imputer_step = ColumnTransformer(
    [
        ('SimpleImputer', SimpleImputer(strategy='most_frequent'), cat_cols),
        ('IterativeImputer', IterativeImputer(estimator=ExtraTreesRegressor(n_estimators=50,max_depth=5),min_value=0), num_cols)
    ],
    remainder='passthrough', 
    verbose_feature_names_out=False
)

num_pipeline = Pipeline([
    ('log', FunctionTransformer(
        func=safe_log1p, 
        inverse_func=np.expm1, 
        validate=False, 
        feature_names_out="one-to-one"
    )),
    ('scaler', RobustScaler())
])

# We dynamically add 'TotalBill' because it was created in the FeatureEngineering step
scaler_step = ColumnTransformer(
    [
        ('num_preprocess', num_pipeline, num_cols + ['TotalBill'])
    ], 
    remainder='passthrough', 
    verbose_feature_names_out=False
)

encoder_step = ColumnTransformer(
    [
        ('OHE', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), ohe_cols),
        ('OrdinalEncoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
        ('DropColumns','drop',['Cabin','Name','PassengerId'])
    ],
    remainder='passthrough', 
    verbose_feature_names_out=False
)

# ---------------------------------------------------------
# 5. The Final Master Pipeline
# ---------------------------------------------------------
master_pipeline = Pipeline([
    ('string_extractor', StringExtractionTransformer()),
    ('imputer', imputer_step),
    ('feature_engineer', FeatureEngineeringTransformer()),
    ('scaler', scaler_step),
    ('encoder', encoder_step)
])

# Example Usage:
X_train_processed = master_pipeline.fit_transform(X_train)
# X_test_processed = master_pipeline.transform(X_test)

C:\Users\Wajih\anaconda3\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [23]:
X_train_processed.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5824 entries, 4696 to 7270
Data columns (total 33 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   HomePlanet_Europa          5824 non-null   float64
 1   HomePlanet_Mars            5824 non-null   float64
 2   Destination_PSO J318.5-22  5824 non-null   float64
 3   Destination_TRAPPIST-1e    5824 non-null   float64
 4   Deck_B                     5824 non-null   float64
 5   Deck_C                     5824 non-null   float64
 6   Deck_D                     5824 non-null   float64
 7   Deck_E                     5824 non-null   float64
 8   Deck_F                     5824 non-null   float64
 9   Deck_G                     5824 non-null   float64
 10  Deck_T                     5824 non-null   float64
 11  AgeGroup_Child             5824 non-null   float64
 12  AgeGroup_Senior            5824 non-null   float64
 13  AgeGroup_Teenager          5824 non-null   float64

In [24]:
import joblib

In [43]:
model = joblib.load('../models/best_model.joblib')

C:\Users\Wajih\anaconda3\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [44]:
test_processed = master_pipeline.transform(test)

In [45]:
test_prediction = model.predict(test_processed)

In [46]:
test_prediction

array([ True, False,  True, ...,  True,  True,  True])

In [47]:
test

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4272,9266_02,Earth,True,G/1496/S,TRAPPIST-1e,34.0,False,0.0,0.0,0.0,0.0,0.0,Jeron Peter
4273,9269_01,Earth,False,NaN,TRAPPIST-1e,42.0,False,0.0,847.0,17.0,10.0,144.0,Matty Scheron
4274,9271_01,Mars,True,D/296/P,55 Cancri e,NaN,False,0.0,0.0,0.0,0.0,0.0,Jayrin Pore
4275,9273_01,Europa,False,D/297/P,NaN,NaN,False,0.0,2680.0,0.0,0.0,523.0,Kitakan Conale


In [49]:
pd.DataFrame(index=test['PassengerId'],columns=['Transported'],data=test_prediction).reset_index().to_csv('Kaggle_Submission.csv',index=False)